# Phase 2 – Part B: Structured Prompt Engineering & Generative AI

## 1. Generative AI Model Selection & Setup

We selected a GPT-based Generative AI model to transform machine-learning outputs into clear, executive-ready screening summaries.

The Generative AI component does not perform prediction or diagnosis. Instead, it receives the supervised model prediction, prediction confidence, important indicators, and clustering context, then generates a concise decision-support explanation.

In [3]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "openai"
])

print("OpenAI package installed successfully.")

OpenAI package installed successfully.


## 1. Environment Setup

This section imports the required libraries and loads the OpenAI API key securely from the environment variables.

The API key is not stored directly in the notebook to avoid exposing sensitive credentials in the GitHub repository.

In [4]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY was not found in the .env file.")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq client is ready.")

OpenAI client is ready.


## 2. OpenAI API Configuration

The OpenAI API key is loaded securely from a local `.env` file.
The key is not stored directly in the notebook and is excluded from the GitHub repository.

In [10]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

client = OpenAI(api_key=api_key)

print("OpenAI client is ready.")

OpenAI client is ready.


## 3. Prompt Template 1 – Individual Patient Risk Briefing

This prompt template is designed to generate an executive-friendly briefing for an individual patient.

It receives the supervised learning prediction, prediction confidence, important contributing indicators, and the patient's clustering context.

The Generative AI model is used only to explain existing machine learning outputs. It does not independently predict or diagnose diabetes.

In [6]:
PROMPT_TEMPLATE_1 = """
You are an assistant for a healthcare screening decision-support system.

Your role is to explain outputs produced by existing machine learning models.
Do not make an independent medical diagnosis.
Do not introduce facts that are not supported by the provided information.

Patient screening information:

Supervised model output:
- Screening result: {prediction}
- Prediction confidence: {probability}

Key contributing indicators:
{top_features}

Clustering output:
- Risk segment: {cluster_segment}
- Cluster profile: {cluster_profile}

Generate a concise individual patient briefing with these sections:

1. Screening Summary
2. Key Contributing Indicators
3. Risk Segment Context
4. Suggested Follow-up Considerations

Requirements:
- Use clear, non-technical language.
- Do not describe the result as a confirmed diagnosis.
- Do not invent medical information.
- Clearly distinguish the individual prediction confidence from the cluster-level profile.
- Keep the response concise and suitable for decision support.
"""

## 4. Test Case 1 – High-Risk Patient

The first test case represents a patient with a high supervised-model prediction confidence and a high-risk clustering profile.

This case is used to evaluate whether the prompt can clearly explain a high-risk screening result without presenting it as a confirmed medical diagnosis.

In [7]:
test_case_1 = {
    "prediction": "Further clinical assessment recommended",
    "probability": "84%",
    "top_features": "HighBP, BMI, GenHlth",
    "cluster_segment": "High risk",
    "cluster_profile": (
        "Patients with poor general health, many days of poor physical "
        "and mental health, difficulty walking, low physical activity, "
        "lower income, and higher rates of heart disease and stroke."
    )
}

## 5. Constructing the Prompt for Test Case 1

The values from Test Case 1 are inserted into the placeholders defined in Prompt Template 1.

This creates the final prompt that will be sent to the Generative AI model.

In [8]:
prompt_case_1 = PROMPT_TEMPLATE_1.format(**test_case_1)

print(prompt_case_1)


You are an assistant for a healthcare screening decision-support system.

Your role is to explain outputs produced by existing machine learning models.
Do not make an independent medical diagnosis.
Do not introduce facts that are not supported by the provided information.

Patient screening information:

Supervised model output:
- Screening result: Further clinical assessment recommended
- Prediction confidence: 84%

Key contributing indicators:
HighBP, BMI, GenHlth

Clustering output:
- Risk segment: High risk
- Cluster profile: Patients with poor general health, many days of poor physical and mental health, difficulty walking, low physical activity, lower income, and higher rates of heart disease and stroke.

Generate a concise individual patient briefing with these sections:

1. Screening Summary
2. Key Contributing Indicators
3. Risk Segment Context
4. Suggested Follow-up Considerations

Requirements:
- Use clear, non-technical language.
- Do not describe the result as a confirmed

## 6. Generative AI API Call

The completed prompt is sent to the selected Generative AI model through the API.

The model generates an executive-friendly explanation based only on the supplied machine learning outputs and clustering information.

In [13]:
response = client.responses.create(
    model="openai/gpt-oss-20b",
    input=prompt_case_1
)

template1_case1_output = response.output_text

print(template1_case1_output)

AuthenticationError: Error code: 401 - Incorrect API key provided: sk-proj-********************************************************************************************************************************************************54cA. You can find your API key at https://platform.openai.com/account/api-keys.

## 7. Saving the Generated Output

The generated response is saved as a text file so that the output can be reviewed later and compared with the outputs produced by the second prompt template.

Saving the outputs also allows the project results to be reviewed without requiring the API to be called again.

from pathlib import Path

output_dir = Path("example_outputs")
output_dir.mkdir(exist_ok=True)

with open(
    output_dir / "template1_case1.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(template1_case1_output)

print("Case 1 saved successfully.")